In [1]:
import sys
import os
import urllib3
from configparser import ConfigParser

# Add your local ThreatConnect SDK to path
sys.path.append("Z:/HTOC/Data_Analytics/threatconnect")
from ThreatConnect import ThreatConnect
from RequestObject import RequestObject
from Owners import Owners

# Add your project repo to path
project_root = r"C:\Users\jaskew\Documents\project_repository\scripts\Data Movement\ThrearConnect-api-pull"
if project_root not in sys.path:
    sys.path.append(project_root)

from utils.config_loader import load_config

# Load API config
config_path = os.path.join(project_root, "utils", "config.json")
try:
    api_secret_key, api_access_id, api_base_url, api_default_org = load_config(config_path)
    print(f"Loaded config from: {config_path}")
    print(f"Base URL: {api_base_url}")
    print(f"Access ID: {api_access_id}")
    print(f"Default Org: {api_default_org}")
except Exception as e:
    print(f"[ERROR] Failed to load configuration: {e}")
    sys.exit(1)

# Disable SSL verification warnings (use cautiously)
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)
verify_ssl = False

# Initialize ThreatConnect session
try:
    tc = ThreatConnect(api_access_id, api_secret_key, api_default_org, api_base_url)
    print("ThreatConnect initialized.")
except Exception as e:
    print(f"[ERROR] Failed to initialize ThreatConnect: {e}")
    sys.exit(1)

# Define the owner (organization scope)
owner = 'HTOC Org'

# Create a request object to fetch indicators (or other data)
try:
    ro = RequestObject()
    ro.set_http_method('GET')
    ro.set_owner(owner)
    ro.set_owner_allowed(True)
    # ro.set_resource_pagination(True)  # Uncomment if needed
    print("RequestObject successfully created.")
except Exception as e:
    print(f"[ERROR] Failed to initialize RequestObject: {e}")
    sys.exit(1)



Loaded config from: C:\Users\jaskew\Documents\project_repository\scripts\Data Movement\ThrearConnect-api-pull\utils\config.json
Base URL: https://hvs.threatconnect.com/api
Access ID: 09783848890162390382
Default Org: HTOC Org
ThreatConnect initialized.
RequestObject successfully created.


In [2]:
import pandas as pd
from datetime import datetime, timedelta
from collections import defaultdict
import pytz

# Define time period
# Calculate the start date (2 days ago) at 00:00:00 UTC
start_date = (datetime.now(pytz.UTC) - timedelta(days=2)).date()

# Format as 'YYYY-MM-DDT00:00:00Z'
start = f"{start_date}T00:00:00Z"

# List of owners to pull from
import urllib.parse

list_of_owners = ['HTOC Org']
final_results = []
type_names = [
    "Address", "EmailAddress", "File", "Host", "URL", "ASN", "CIDR", "Email Subject", "Hashtag", "Mutex", "Registry Key", "Stripped URL", "User Agent"]
type_name_condition = ", ".join([f'"{t}"' for t in type_names])

for owner in list_of_owners:
    print(f"\nQuerying owner: {owner}")
    try:
        tql_raw = (
            f'ownerName EQ "{owner}" AND '
            f'typeName IN ({type_name_condition})'
            f'lastObserved >= "{start}"'
        )
        tql_encoded = urllib.parse.quote(tql_raw)

        ro.set_http_method('GET')
        ro.set_request_uri(
            f'/v3/indicators?tql={tql_encoded}&fields=tags,observations&resultStart=0&resultLimit=10000'
        )

         # Send the request
        response = tc.api_request(ro)

        # Parse response
        if response.headers.get('content-type') == 'application/json':
            results = response.json()
            final_results.append(results)
        else:
            print(f"Unexpected response format: {response.headers.get('content-type')}")

    except Exception as e:
        print(f"Failed to query indicators for {owner}: {e}")

# Normalize and clean results
if final_results:
    # Extract and normalize data only if 'data' key exists and contains 'summary'
    normalized_data = []
    for result in final_results:
        if 'data' in result:
            for item in result['data']:
                if 'summary' in item:
                    normalized_data.append(item)

    if normalized_data:
        observed_src = pd.json_normalize(normalized_data)
        observed_src['indicator'] = observed_src['summary'].str.split(' ', expand=True)[0].str.strip()
        observed_src = observed_src.drop_duplicates(subset='indicator', keep='first')  # Remove duplicates based on 'indicator'
        observed_src = observed_src[observed_src['lastObserved'] >= start]
        print(f"\nRetrieved {len(observed_src)} unique observed indicators.")
    else:
        print("\nNo valid indicators with 'summary' key retrieved.")
else:
    print("\nNo indicators retrieved.")




Querying owner: HTOC Org

Retrieved 296 unique observed indicators.


In [3]:
observed_src

,id,dateAdded,ownerId,ownerName,webLink,type,lastModified,rating,confidence,summary,...,source,sha256,url,text,address,md5,sha1,Subject,size,indicator
0,5629499536037128,2025-04-14T16:04:29Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-05-16T12:21:53Z,5.0,97.0,171.25.193.235,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,171.25.193.235
1,5629499537015449,2025-04-23T15:01:06Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-05-16T12:21:39Z,5.0,97.0,206.217.206.68,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,206.217.206.68
2,5629499541089762,2025-05-14T12:23:45Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-05-16T12:21:25Z,3.0,80.0,193.163.125.226,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,193.163.125.226
3,5629499541090447,2025-05-14T17:44:45Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-05-16T12:21:11Z,3.0,80.0,87.236.176.15,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,87.236.176.15
4,5629499541090448,2025-05-14T17:44:45Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-05-16T12:20:56Z,3.0,80.0,104.152.52.237,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,104.152.52.237
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9588,234703,2017-01-03T15:27:15Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-05-13T06:12:39Z,5.0,97.0,185.65.134.76,...,Department of Homeland Security \r\nNCCIC-- US...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,185.65.134.76
9728,4515497,2024-02-01T17:29:04Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-05-13T06:12:37Z,3.0,32.0,68.71.254.6,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,68.71.254.6
9812,4554083,2024-03-29T14:31:35Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-05-13T06:12:36Z,3.0,14.0,45.83.220.203,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,45.83.220.203
9814,4553546,2024-03-29T13:13:10Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-05-13T06:12:36Z,3.0,6.0,169.150.226.164,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,169.150.226.164


In [4]:
import os
import pandas as pd
from datetime import datetime, timedelta

# Base file path with placeholder for date
base_path = r"Z:/HTOC/Data_Analytics/Data/OpDiv_Observations/htoc_opdiv_obs_d{date}.csv"
#base_path = r"C:\Users\jaskew\Documents\project_repository\data\raw\ObservationDataFiles\htoc_opdiv_obs_d{date}.csv"
date_format = "%Y%m%d"

def get_file_paths(base_path, days=3):
    """ Generate file paths for the last `days` days using list comprehension. """
    today = datetime.utcnow()
    dates_to_pull = [(today - timedelta(days=i)).strftime(date_format) for i in range(days)]
    
    # Construct file paths
    file_paths = [base_path.format(date=dt) for dt in dates_to_pull]
    
    # Filter for existing files
    existing_files = [file_path for file_path in file_paths if os.path.exists(file_path)]
    
    if not existing_files:
        print("No files found for the specified date range.")
    else:
        print(f"Files to be loaded: {existing_files}")
    
    return existing_files

def load_observed_data(file_paths):
    """ Load and concatenate observed data from multiple files. """
    data_frames = []

    for file_path in file_paths:
        try:
            df = pd.read_csv(file_path)
            data_frames.append(df)
        except Exception as e:
            print(f"Error reading file {file_path}: {e}")
    
    # Concatenate data
    if data_frames:
        observed_data_df = pd.concat(data_frames, ignore_index=True)
        print(f"Loaded data from {len(data_frames)} files.")
    else:
        observed_data_df = pd.DataFrame()

    return observed_data_df

# Example Usage:
# Fetch file paths for the last 3 days
file_paths = get_file_paths(base_path, days=3)

# Load observed data
observed_data_df = load_observed_data(file_paths)



Files to be loaded: ['Z:/HTOC/Data_Analytics/Data/OpDiv_Observations/htoc_opdiv_obs_d20250516.csv', 'Z:/HTOC/Data_Analytics/Data/OpDiv_Observations/htoc_opdiv_obs_d20250515.csv', 'Z:/HTOC/Data_Analytics/Data/OpDiv_Observations/htoc_opdiv_obs_d20250514.csv']
Loaded data from 3 files.


In [22]:
import pandas as pd
from datetime import timedelta, date

# Define cutoff time in UTC
cutoff = pd.Timestamp.utcnow()
date_added_cutoff = cutoff - timedelta(days=30)

# Initialize an empty DataFrame to store filtered tags
filtered_tags = pd.DataFrame()

# Loop through each row in observed_src
for _, row in observed_src.iterrows():
    tags_data = row.get('tags.data')
    
    if isinstance(tags_data, list):
        # Normalize the tags data for the current row
        tags = pd.json_normalize(tags_data)

        # Ensure 'name' column is of string type
        tags['name'] = tags['name'].astype(str)

        # Create a new column with a list of all tag names
        all_tags_list = tags['name'].tolist()

        # Filter for "API" tags
        api_tags = tags[tags['name'].str.contains('API', case=False, na=False)].copy()

        if not api_tags.empty:
            # Add metadata columns and all_tags list as a single value (not as a series)
            for col in ['summary', 'observations', 'description', 'type', 'dateAdded', 'lastModified', 'lastObserved']:
                api_tags[col] = row.get(col)
            
            # Assign the all_tags list as a single value for the entire set of API tags
            api_tags['all_tags'] = [all_tags_list] * len(api_tags)

            # Append to the filtered DataFrame
            filtered_tags = pd.concat([filtered_tags, api_tags], ignore_index=True)

# Ensure 'lastObserved' is datetime
filtered_tags['lastObserved'] = pd.to_datetime(filtered_tags['lastObserved'], errors='coerce')
filtered_tags['dateAdded'] = pd.to_datetime(filtered_tags['dateAdded'], errors='coerce')

# Ensure necessary columns exist
required_columns = ['indicator', 'OpDiv', 'obs_date']
missing_columns = [col for col in required_columns if col not in observed_data_df.columns]

if missing_columns:
    raise ValueError(f"Missing columns in ProcessedObservedData.csv: {missing_columns}")

# Clean the 'name' column by removing ' Splunk API'
filtered_tags['cleaned_name'] = filtered_tags['name'].str.replace(r'\s+Splunk API$', '', regex=True)

# Initialize the observed_date column with NaN
filtered_tags['observed_date'] = None

# Iterate through each row and search for matching indicator and opdiv
for index, row in filtered_tags.iterrows():
    # Extract summary and cleaned_name
    summary = row['summary']
    cleaned_name = row['cleaned_name']
    
    # Search for matching rows in the observed data
    match = observed_data_df[(observed_data_df['indicator'] == summary) & (observed_data_df['OpDiv'] == cleaned_name)]
    
    # If a match is found, extract the obs_date
    if not match.empty:
        # Assign the first matching obs_date
        filtered_tags.at[index, 'observed_date'] = match['obs_date'].iloc[0]

filtered_tags['observed_date'] = pd.to_datetime(filtered_tags['observed_date'], errors='coerce')

# Drop the 'cleaned_name' column as it is no longer needed
filtered_tags.drop(columns=['cleaned_name'], inplace=True)

# Filter to last 48 hours
recent_tags = filtered_tags[filtered_tags['lastObserved'] >= cutoff - timedelta(hours=48)].copy()

# Make `cutoff` a naive datetime to match the naive `observed_date`
cutoff_naive = cutoff.tz_convert(None)

# Apply the filter directly
recent_tags = filtered_tags[filtered_tags['observed_date'] >= cutoff_naive - timedelta(days=2)].copy()
# Extract partner names and remove ' Splunk API' suffix
recent_tags['partner'] = recent_tags['name'].str.replace(' Splunk API', '', regex=False)

# Group partners per summary
partner_groups = (
    recent_tags.groupby('summary')['partner']
    .agg(['nunique', lambda x: ', '.join(sorted(set(x)))]).reset_index()
    .rename(columns={'nunique': 'partner_count', '<lambda_0>': 'partners'})
)

# Merge partner information back into recent_tags
recent_tags = recent_tags.merge(partner_groups, on='summary', how='left')

# Apply additional filters
recent_tags = recent_tags[recent_tags['partner_count'] >= 2]
recent_tags = recent_tags[recent_tags['observations'] < 15000]

# Filter by dateAdded in the last 30 days
recent_tags = recent_tags[recent_tags['dateAdded'] >= date_added_cutoff]

# Drop duplicates based on 'summary'
recent_tags = recent_tags.drop_duplicates(subset='summary', keep='first')

# Drop rows where 'description' is equal to 'RITM0580258/TASK0875884' because of NIH SOAR
recent_tags = recent_tags[recent_tags['description'] != 'RITM0580258/TASK0875884']

# Drop unnecessary columns
columns_to_drop = [
    'techniqueId', 'tactics.data', 'tactics.count',
    'platforms.data', 'platforms.count', 'partner', 'name'
]
recent_tags = recent_tags.drop(columns=[col for col in columns_to_drop if col in recent_tags.columns], errors='ignore')


# Remove rows where 'all_tags' contains 'I&W'
recent_tags = recent_tags[~recent_tags['all_tags'].apply(lambda x: 'I&W' in x)]

recent_tags


,id,description,lastUsed,summary,observations,type,dateAdded,lastModified,lastObserved,all_tags,observed_date,partner_count,partners
10,35760,FBI Email Alert May 14 2025,2025-05-16T12:21:53Z,87.236.176.15,54,Address,2025-05-14 17:44:45+00:00,2025-05-16T12:21:11Z,2025-05-16 00:00:00+00:00,"[OS Splunk API, VA OIS CSOC CTS Splunk, VA CSO...",2025-05-15,2,"HRSA, OS"
17,35760,INC9055547,2025-05-16T12:21:53Z,36.138.156.96,9036,Address,2025-05-13 18:36:26+00:00,2025-05-16T12:20:29Z,2025-05-15 00:00:00+00:00,"[OS Splunk API, FDA Splunk API, CMS Splunk API...",2025-05-15,5,"CMS, FDA, HRSA, NIH, OS"
23,35760,FBI Email Alert May 14 2025,2025-05-16T12:21:53Z,104.156.155.30,2342,Address,2025-05-14 17:44:47+00:00,2025-05-16T12:20:01Z,2025-05-16 00:00:00+00:00,"[OS Splunk API, VA OIS CSOC CTS Splunk, FDA Sp...",2025-05-16,5,"CMS, FDA, HRSA, IHS, OS"
42,35760,FBI Email Alert May 14 2025,2025-05-16T12:21:53Z,87.236.176.78,44,Address,2025-05-14 17:44:48+00:00,2025-05-16T12:18:23Z,2025-05-16 00:00:00+00:00,"[OS Splunk API, VA OIS CSOC CTS Splunk, FDA Sp...",2025-05-15,2,"FDA, OS"
112,35760,FBI Email Alert May 14 2025,2025-05-16T12:21:53Z,87.236.176.93,59,Address,2025-05-14 17:44:48+00:00,2025-05-16T12:12:20Z,2025-05-16 00:00:00+00:00,"[OS Splunk API, VA OIS CSOC CTS Splunk, VA CSO...",2025-05-15,2,"HRSA, OS"
142,35057,INC9052265,2025-05-16T12:21:53Z,36.111.204.0,9345,Address,2025-05-12 12:36:04+00:00,2025-05-16T12:10:15Z,2025-05-15 00:00:00+00:00,"[FDA Splunk API, CMS Splunk API, HRSA Splunk A...",2025-05-15,3,"CMS, FDA, HRSA"
163,35057,INC9056503,2025-05-16T12:21:53Z,42.51.3.33,7991,Address,2025-05-14 13:39:28+00:00,2025-05-16T12:08:37Z,2025-05-16 00:00:00+00:00,"[FDA Splunk API, CMS Splunk API, HRSA Splunk A...",2025-05-16,3,"CMS, FDA, HRSA"
236,35057,FBI Email Alert May 14 2025,2025-05-16T12:21:53Z,87.236.176.176,30,Address,2025-05-14 17:44:42+00:00,2025-05-16T12:03:12Z,2025-05-16 00:00:00+00:00,"[VA OIS CSOC CTS Splunk, FDA Splunk API, VA CS...",2025-05-15,2,"FDA, HRSA"
238,35057,FBI Email Alert May 14 2025,2025-05-16T12:21:53Z,104.152.52.110,28,Address,2025-05-14 17:44:46+00:00,2025-05-16T12:02:44Z,2025-05-16 00:00:00+00:00,"[VA OIS CSOC CTS Splunk, FDA Splunk API, CMS S...",2025-05-16,2,"CMS, FDA"
240,23667,FBI Email Alert May 14 2025,2025-05-16T12:21:53Z,87.236.176.223,45,Address,2025-05-14 17:44:50+00:00,2025-05-16T12:02:30Z,2025-05-16 00:00:00+00:00,"[VA OIS CSOC CTS Splunk, VA CSOC CTS Splunk, H...",2025-05-15,2,"HRSA, NIH"


In [42]:
import pandas as pd
import urllib.parse

# Extract unique indicators from recent_tags
indicators = recent_tags['summary'].unique()

# Initialize a list to store attribute data
attributes_data = []

# Iterate over unique indicators
for indicator in indicators:
    try:
        # API Request
        ro.set_http_method('GET')
        ro.set_request_uri(f'/v3/indicators/{indicator}?fields=attributes&resultStart=0&resultLimit=1000')
        response = tc.api_request(ro)

        # Parse JSON response
        if response.headers.get('content-type') == 'application/json':
            data = response.json().get('data', {})
            
            # Extract attributes data
            for attr in data.get('attributes', {}).get('data', []):
                attr.update({
                    'id': data.get('id'),
                    'summary': data.get('summary'),
                    'type': data.get('type'),
                    'ownerName': data.get('ownerName')
                })
                attributes_data.append(attr)

    except Exception:
        pass  # Silent fail to keep processing

# Create attributes DataFrame
attributes_observed_src = pd.DataFrame(attributes_data)

# Un-nest 'createdBy' and filter out 'SOAR' entries
if not attributes_observed_src.empty and 'createdBy' in attributes_observed_src.columns:
    created_by_df = pd.json_normalize(attributes_observed_src.pop('createdBy'))
    attributes_observed_src = pd.concat([attributes_observed_src, created_by_df], axis=1)
    attributes_observed_src = attributes_observed_src[attributes_observed_src['lastName'] != 'SOAR']

# Drop duplicates based on 'id'
attributes_observed_src = attributes_observed_src.drop_duplicates(subset='id').reset_index(drop=True)

# Filter recent_tags to include only indicators in attributes_observed_src
filtered_recent_tags = recent_tags[recent_tags['summary'].isin(attributes_observed_src['summary'])].reset_index(drop=True)


In [43]:
filtered_recent_tags

,id,description,lastUsed,summary,observations,type,dateAdded,lastModified,lastObserved,all_tags,observed_date,partner_count,partners
0,35057,INC9052265,2025-05-16T12:21:53Z,36.111.204.0,9345,Address,2025-05-12 12:36:04+00:00,2025-05-16T12:10:15Z,2025-05-15 00:00:00+00:00,"[FDA Splunk API, CMS Splunk API, HRSA Splunk A...",2025-05-15,3,"CMS, FDA, HRSA"


In [39]:
import pandas as pd
import requests
import time
import ipaddress
from datetime import datetime
import concurrent.futures

# API Keys
VT_API_KEY = "a8b3e24dbd2e6c0cb002784aeb8fee6217a6a425cb11ddf9a3d3361281fbbb08"
OTX_API_KEY = "ea83cf4792fc5db7acc741e82286c0b717fc99be98ec5b14de7e63cd3f74bcfe"

# Headers for API requests
VT_HEADERS = {"x-apikey": VT_API_KEY}
OTX_HEADERS = {"X-OTX-API-KEY": OTX_API_KEY}

# API URLs
VT_URL_IP = "https://www.virustotal.com/api/v3/ip_addresses/{}"
VT_URL_DOMAIN = "https://www.virustotal.com/api/v3/domains/{}"
OTX_URL_IP = "https://otx.alienvault.io/api/v1/indicators/IPv4/{}/general"
OTX_URL_DOMAIN = "https://otx.alienvault.io/api/v1/indicators/domain/{}/general"
OTX_URL_HOSTNAME = "https://otx.alienvault.io/api/v1/indicators/hostname/{}"

def is_ip(value):
    """ Check if the given value is a valid IP address. """
    try:
        ipaddress.ip_address(value)
        return True
    except ValueError:
        return False

def determine_query_type(query):
    """ Determine if the query is an IP, domain, or hostname. """
    if is_ip(query):
        return "ip"
    elif "." in query:
        return "hostname"
    else:
        return "domain"

def fetch_virustotal_data(query):
    """Fetch data from VirusTotal API for IP or Domain."""
    query_type = determine_query_type(query)
    url = VT_URL_IP.format(query) if query_type == "ip" else VT_URL_DOMAIN.format(query)

    try:
        response = requests.get(url, headers=VT_HEADERS, verify=False)
        response.raise_for_status()
        data = response.json().get("data", {}).get("attributes", {})

        return {
            "search_term": query,
            "asn": data.get('asn'),
            "as_owner": data.get('as_owner'),
            "country": data.get('country'),
            "network": data.get('network'),
            "last_analysis_stats": data.get('last_analysis_stats'),
            "reputation": data.get('reputation'),
            "total_votes": data.get('total_votes'),
            "open_ports": [s.get("port") for s in data.get("services", []) if "port" in s],
            "link": f"https://www.virustotal.com/gui/ip-address/{query}" if query_type == "ip" else f"https://www.virustotal.com/gui/domain/{query}"
        }

    except Exception as e:
        print(f"VirusTotal Error for {query}: {e}")
        return None

def fetch_otx_data(query):
    """Fetch data from OTX API for IP, Domain, or Hostname."""
    query_type = determine_query_type(query)

    if query_type == "ip":
        url = OTX_URL_IP.format(query)
    elif query_type == "hostname":
        url = OTX_URL_HOSTNAME.format(query)
    else:
        url = OTX_URL_DOMAIN.format(query)

    try:
        response = requests.get(url, headers=OTX_HEADERS, verify=False)
        response.raise_for_status()
        data = response.json()

        return {
            "search_term": query,
            "base_indicator": data.get('base_indicator', {}),
            "reputation": data.get('reputation'),
            "geo": data.get('geo', {}),
            "whois": data.get('whois', {}),
            "open_ports": data.get('open_ports', []),
            "link": f"https://otx.alienvault.com/indicator/{query_type}/{query}"
        }

    except Exception as e:
        print(f"OTX Error for {query}: {e}")
        return None

def unnest_base_indicator(df):
    """ Unnest the 'base_indicator' column and create separate columns for its keys. """
    if 'base_indicator' not in df.columns:
        return df

    base_df = pd.json_normalize(df['base_indicator'])
    base_df.columns = [f"base_{col}" for col in base_df.columns]

    df = df.drop(columns=['base_indicator']).reset_index(drop=True)
    df = pd.concat([df, base_df], axis=1)
    return df

def process_indicator(indicator, observed_by, partner_count):
    """Fetch data for a single indicator."""
    timestamp = datetime.utcnow().strftime("%Y-%m-%d %H:%M:%S")

    with concurrent.futures.ThreadPoolExecutor() as executor:
        vt_future = executor.submit(fetch_virustotal_data, indicator)
        otx_future = executor.submit(fetch_otx_data, indicator)

        vt_data = vt_future.result()
        otx_data = otx_future.result()

    if vt_data:
        vt_data.update({
            "timestamp": timestamp,
            "observed_by": observed_by,
            "partner_count": partner_count
        })

    if otx_data:
        otx_data.update({
            "timestamp": timestamp,
            "observed_by": observed_by,
            "partner_count": partner_count
        })

    return vt_data, otx_data

def main(recent_tags):
    """ Main function to process indicators. """
    if 'summary' not in recent_tags.columns:
        print("The 'summary' column is missing.")
        return pd.DataFrame(), pd.DataFrame()

    search_terms = recent_tags['summary'].dropna().unique().tolist()
    print(f"Processing {len(search_terms)} unique search terms.")

    vt_results = []
    otx_results = []

    with concurrent.futures.ThreadPoolExecutor() as executor:
        futures = []
        for indicator in search_terms:
            partners = recent_tags.loc[recent_tags['summary'] == indicator, 'partners'].values
            partner_count = recent_tags.loc[recent_tags['summary'] == indicator, 'partner_count'].values
            observed_by = partners[0] if len(partners) > 0 else "N/A"
            partner_count = partner_count[0] if len(partner_count) > 0 else "N/A"

            futures.append(executor.submit(process_indicator, indicator, observed_by, partner_count))

        for future in concurrent.futures.as_completed(futures):
            vt_data, otx_data = future.result()
            if vt_data:
                vt_results.append(vt_data)
            if otx_data:
                otx_results.append(otx_data)

    vt_df = pd.DataFrame(vt_results)
    otx_df = pd.DataFrame(otx_results)

    otx_df = unnest_base_indicator(otx_df)

    return vt_df, otx_df

if __name__ == "__main__":
    vt_df, otx_df = main(filtered_recent_tags)
    print("Script completed successfully.")


Processing 1 unique search terms.
Script completed successfully.


In [40]:
vt_df

,search_term,asn,as_owner,country,network,last_analysis_stats,reputation,total_votes,open_ports,link,timestamp,observed_by,partner_count
0,36.111.204.0,63835,"No.293,Wanbao Avenue",CN,36.111.200.0/21,"{'malicious': 8, 'suspicious': 3, 'undetected'...",-1,"{'harmless': 0, 'malicious': 1}",[],https://www.virustotal.com/gui/ip-address/36.1...,2025-05-16 12:40:17,"CMS, FDA, HRSA",3


In [41]:
import os
import pandas as pd
from datetime import datetime
from docx import Document

# File paths
#TEMPLATE_PATH = r"z:\HTOC\HTOC Reports\I&W Reports\5. I&W Staging\I&W Report Template.docx"
TEMPLATE_PATH = r"C:\Users\jaskew\Documents\project_repository\notebooks\I&W Reporting\I&W Report Template.docx"
#OUTPUT_DIR = r"z:\HTOC\HTOC Reports\I&W Reports\5. I&W Staging\Generated Reports"
OUTPUT_DIR = r"C:\Users\jaskew\Documents\project_repository\notebooks\I&W Reporting\Generated_reports"

# Ensure output directory exists
os.makedirs(OUTPUT_DIR, exist_ok=True)

def consolidate_sources(vt_df, otx_df):
    """ Consolidate links from both DataFrames for each search term. """
    # Collect links from VirusTotal
    vt_links = vt_df.groupby('search_term')['link'].apply(lambda x: ', '.join(x.dropna())).reset_index()
    vt_links.columns = ['search_term', 'vt_links']

    # Collect links from OTX
    otx_links = otx_df.groupby('search_term')['link'].apply(lambda x: ', '.join(x.dropna())).reset_index()
    otx_links.columns = ['search_term', 'otx_links']

    # Merge the two link sets
    consolidated = pd.merge(vt_links, otx_links, on='search_term', how='outer')

    # Combine the links, handling NaN values
    consolidated['sources'] = consolidated[['vt_links', 'otx_links']].fillna('').apply(
        lambda x: ', '.join(filter(None, x)), axis=1
    )

    return consolidated[['search_term', 'sources']]

def extract_date(timestamp):
    """ Extract only the date from the timestamp. """
    if pd.isna(timestamp) or timestamp == 'N/A':
        return 'N/A'
    
    # Handle datetime object or string
    try:
        # Attempt to parse as a datetime object
        if isinstance(timestamp, str):
            timestamp = datetime.strptime(timestamp, "%Y-%m-%d %H:%M:%S")
        return timestamp.strftime("%Y-%m-%d")
    except Exception:
        return 'N/A'

def populate_table(table, data):
    """ Populate a Word table with the given data. """
    # Iterate over data and populate rows
    for index, row in data.iterrows():
        # Insert a new row before the last row (template row)
        new_row = table.add_row().cells
        new_row[0].text = str(row.get('search_term', 'N/A'))
        new_row[1].text = str(row.get('type', 'N/A'))
        new_row[2].text = extract_date(row.get('observed_date', 'N/A'))
        new_row[3].text = str(row.get('observed_by_otx', 'N/A'))
        new_row[4].text = str(row.get('notes', ''))
        
def fill_word_template(template_path, output_path, df):
    """ Fill the template with data and place sources outside the table. """
    if not os.path.exists(template_path):
        print(f"Template not found: {template_path}")
        return

    try:
        doc = Document(template_path)

        # Populate the table
        table = None
        for tbl in doc.tables:
            if "Indicators/Identifiers" in tbl.rows[0].cells[0].text:
                table = tbl
                break

        if table:
            populate_table(table, df)

        # Find and replace `{{sources}}` placeholder outside the table
        for para in doc.paragraphs:
            if "{{sources}}" in para.text:
                all_sources = ', '.join(df['sources'].dropna().unique())
                para.text = para.text.replace("{{sources}}", all_sources)

        # Save the modified document
        doc.save(output_path)
        print(f"Saved report: {output_path}")

    except Exception as e:
        print(f"Error while generating report: {e}")

def main(vt_df, otx_df, recent_tags):
    """ Main function to handle report generation. """
    # Combine vt_df and otx_df on 'search_term'
    combined_df = pd.merge(vt_df, otx_df, on='search_term', how='outer', suffixes=('_vt', '_otx'))

    # Consolidate sources into a single column
    sources_df = consolidate_sources(vt_df, otx_df)
    combined_df = pd.merge(combined_df, sources_df, on='search_term', how='left')

    # Merge recent_tags using left_on and right_on, drop 'summary' afterward to avoid duplicates
    if not recent_tags.empty:
        combined_df = pd.merge(
            combined_df,
            recent_tags[['summary', 'observations', 'type', 'partners', 'observed_date']],
            left_on='search_term',
            right_on='summary',
            how='left'
        )
        # Drop redundant 'summary' column to eliminate duplication
        combined_df.drop(columns=['summary'], inplace=True)

    # Define output file path
    current_date = pd.Timestamp.now().strftime("%Y-%m-%d")
    output_file = os.path.join(OUTPUT_DIR, f"I&W_Report_{current_date}.docx")

    # Fill the template table with data
    fill_word_template(TEMPLATE_PATH, output_file, combined_df)
    
if __name__ == "__main__":
    main(vt_df, otx_df, recent_tags)



Saved report: C:\Users\jaskew\Documents\project_repository\notebooks\I&W Reporting\Generated_reports\I&W_Report_2025-05-16.docx
